# Diagnóstico de calidad de datos: demanda M5, tienda CA_1

Dataset analizado: `../data/processed/obt_sales_CA_1.csv.gz` (tabla `OBT_SALES` del pipeline M5, filtrada a la tienda CA_1; una fila por producto y día). Fecha del diagnóstico: 7 de noviembre de 2026.

El resultado es el diagnóstico documentado, no todavía la limpieza.

In [1]:
import pandas as pd

datos = pd.read_csv("../data/processed/obt_sales_CA_1.csv.gz")
print("Filas y columnas:", datos.shape)
datos.head()

Filas y columnas: (5918109, 40)


,date_key,date,d,d_num,item_id,dept_id,cat_id,store_id,state_id,units,...,date_ly_364,date_key_ly_364,is_snap,is_pre_launch,is_store_closed,is_sales_spike,is_suspected_stockout_conservative,is_suspected_stockout_nb,week_idx,_loaded_at
0,20110129,2011-01-29,d_1,1,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,3,...,2010-01-30,NaN,False,False,False,False,False,False,0,2026-10-21 00:23:53.542
1,20110130,2011-01-30,d_2,2,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,0,...,2010-01-31,NaN,False,False,False,False,False,False,0,2026-10-21 00:23:53.542
2,20110131,2011-01-31,d_3,3,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,0,...,2010-02-01,NaN,False,False,False,False,False,False,0,2026-10-21 00:23:53.542
3,20110201,2011-02-01,d_4,4,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,1,...,2010-02-02,NaN,True,False,False,False,False,False,0,2026-10-21 00:23:53.542
4,20110202,2011-02-02,d_5,5,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,4,...,2010-02-03,NaN,True,False,False,False,False,False,0,2026-10-21 00:23:53.542


## Duplicados

In [2]:
print("Filas duplicadas:", datos.duplicated().sum())
print("Duplicados por (item, fecha):", datos.duplicated(["item_id", "date"]).sum())
datos[datos.duplicated(keep=False)].head(10)

Filas duplicadas: 0


Duplicados por (item, fecha): 0


,date_key,date,d,d_num,item_id,dept_id,cat_id,store_id,state_id,units,...,date_ly_364,date_key_ly_364,is_snap,is_pre_launch,is_store_closed,is_sales_spike,is_suspected_stockout_conservative,is_suspected_stockout_nb,week_idx,_loaded_at


**¿Cuántas filas duplicadas encontraste, y por qué crees que existen?**

Ninguna: 0 filas completas repetidas y 0 repetidas por (producto, fecha). Es lo esperado, porque la tabla tiene exactamente 3 049 productos × 1 941 días = 5 918 109 filas.

**Estrategia y justificación:**

No hay nada que quitar. Dejo la verificación como prueba de calidad en el pipeline, para que avise si una carga futura duplica filas.

## Valores faltantes

In [3]:
faltantes = datos.isna().sum()
print(faltantes[faltantes > 0])
print("Faltantes de sell_price que son pre-lanzamiento:", (datos["sell_price"].isna() & datos["is_pre_launch"]).sum())
print("Faltantes de sell_price que NO son pre-lanzamiento:", (datos["sell_price"].isna() & ~datos["is_pre_launch"]).sum())

# Valores que isna() no detecta: revisa los valores distintos de cada
# columna de texto, buscando marcadores como "N/D", "-" o "sin dato".
for columna in ["event_name_1", "event_type_1", "dept_id", "cat_id", "weekday"]:
    print(columna, datos[columna].unique()[:15])

sell_price         1129842
revenue            1129842
event_name_1       5436367
event_type_1       5436367
event_name_2       5905913
event_type_2       5905913
date_key_ly_364    1109836
dtype: int64
Faltantes de sell_price que son pre-lanzamiento: 1129842
Faltantes de sell_price que NO son pre-lanzamiento: 0
event_name_1 [nan 'SuperBowl' 'ValentinesDay' 'PresidentsDay' 'LentStart' 'LentWeek2'
 'StPatricksDay' 'Purim End' 'OrthodoxEaster' 'Pesach End' 'Cinco De Mayo'
 "Mother's day" 'MemorialDay' 'NBAFinalsStart' 'NBAFinalsEnd']
event_type_1 [nan 'Sporting' 'Cultural' 'National' 'Religious']


dept_id ['FOODS_1' 'FOODS_2' 'FOODS_3' 'HOBBIES_1' 'HOBBIES_2' 'HOUSEHOLD_1'
 'HOUSEHOLD_2']
cat_id ['FOODS' 'HOBBIES' 'HOUSEHOLD']


weekday ['Saturday' 'Sunday' 'Monday' 'Tuesday' 'Wednesday' 'Thursday' 'Friday']


**¿Qué columnas tienen valores faltantes, visibles o disfrazados de texto, y cuántos por columna?**

`sell_price` y `revenue`: 1 129 842 cada una (19.1%); `event_name_1`/`event_type_1`: 5 436 367 (91.9%); `event_name_2`/`event_type_2`: 5 905 913 (99.8%); `date_key_ly_364`: 1 109 836. No hay marcadores de texto disfrazados (`N/D`, `-`, `sin dato`) en las columnas revisadas.

**¿Faltan al azar, o siguen un patrón relacionado con otra variable?**

Siguen un patrón. Los faltantes de precio coinciden exactamente con `is_pre_launch` (1 129 842 filas, ninguna fuera de ellas): son días anteriores al lanzamiento del producto, con 0 unidades y sin precio. Los de `event_*` son días sin evento (no un error). `date_key_ly_364` falta en el primer año, que no tiene "mismo día del año pasado".

**Estrategia y justificación por columna:**

`sell_price`/`revenue`: quitar las filas pre-lanzamiento del entrenamiento (no son demanda, el producto aún no existía). `event_*`: no imputar; los nulos significan "sin evento". `date_key_ly_364`: no la uso.

## Inconsistencias de esquema

In [4]:
print(datos.dtypes)

# Convierte con errors="coerce" lo que deberia ser numerico y revisa las fechas.
precio = pd.to_numeric(datos["sell_price"], errors="coerce")
print("sell_price faltantes tras convertir:", precio.isna().sum(), "(igual que antes:", datos["sell_price"].isna().sum(), ")")
fecha = pd.to_datetime(datos["date"], format="%Y-%m-%d", errors="coerce")
print("fechas que no cumplen AAAA-MM-DD:", fecha.isna().sum(), " rango:", fecha.min().date(), "a", fecha.max().date())
print("unidades negativas:", (datos["units"] < 0).sum(), " precios <= 0:", (datos["sell_price"] <= 0).sum())

date_key                                int64
date                                   object
d                                      object
d_num                                   int64
item_id                                object
dept_id                                object
cat_id                                 object
store_id                               object
state_id                               object
units                                   int64
sell_price                            float64
revenue                               float64
wm_yr_wk                                int64
wm_fiscal_year                          int64
wm_week_of_year                         int64
week_seq                                int64
weekday                                object
wday                                    int64
is_weekend                               bool
day_of_month                            int64
month                                   int64
quarter                           

fechas que no cumplen AAAA-MM-DD: 0  rango: 2011-01-29 a 2016-05-22
unidades negativas: 0  precios <= 0: 0


**¿Alguna columna tiene el tipo de dato incorrecto?**

`date` y `date_ly_364` se leen como texto (`object`) y hay que convertirlas a fecha. Las numéricas (`units`, `sell_price`) tienen el tipo correcto: convertir con `errors="coerce"` no crea faltantes nuevos.

**¿Alguna columna mezcla formatos (fechas, categorías escritas de más de una forma, unidades distintas)?**

No: todas las fechas cumplen `AAAA-MM-DD`, las categorías (`dept_id`, `cat_id`, `weekday`) son consistentes, no hay unidades negativas ni precios menores o iguales a 0.

**Estrategia y justificación:**

Convertir `date` a fecha en el pipeline. Lo que sí requiere cuidado no es el formato sino el significado: de los ceros de `units` (55.2% sin contar pre-lanzamiento), 9.5% (marca conservadora) a 26.5% (modelo binomial negativa) pueden ser quiebres de stock, no falta de demanda, y `revenue` se calcula con `units × sell_price` (fuga de información): no entra como variable de entrada.

## Resumen

**De los problemas encontrados, ¿cuál priorizarías corregir primero, y por qué?**

Las filas pre-lanzamiento (19.1% de la tabla): son ceros sin precio que no son demanda real y distorsionan cualquier promedio o rezago, además de que el modelo aprendería "este producto vende cero". Segundo, la fuga de `revenue` y, tercero, los ceros por quiebre de stock, que reducen la demanda observada.